File version: 1.1

In [1]:
import sys;

print('Python %s on %s' % (sys.version, sys.platform))
sys.path.extend(['../'])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.data_utils import get_connectome

Python 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)] on win32


In [2]:
bnu_series_path = '../data/ts_cut/HCPex/bnu{}.npy'
bnu_labels_path = '../data/ts_cut/HCPex/bnu.csv'
ihb_series_path = '../data/ts_cut/HCPex/ihb.npy'
ihb_labels_path = '../data/ts_cut/HCPex/ihb.csv'

X_bnu = np.concatenate([np.load(bnu_series_path.format(i)) for i in (1, 2)], axis=0)
Y_bnu = pd.read_csv(bnu_labels_path)
X_ihb = np.load(ihb_series_path)
Y_ihb = pd.read_csv(ihb_labels_path)

X_bnu = get_connectome(X_bnu)
X_ihb = get_connectome(X_ihb)

X = np.concatenate([X_bnu, X_ihb])
n_samples = X.shape[0]
X = X.reshape(n_samples, -1).astype('float32')

Y = np.concatenate([Y_bnu, Y_ihb])
X_ihb.shape, Y_ihb.shape

((20, 419, 419), (20, 1))

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

n_samples = X.shape[0]
X = X.reshape(n_samples, -1).astype('float32')
Y = Y.astype('float32')

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2)

class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_size, 128))
        layers.append(nn.ReLU())
        layers.append(nn.Linear(128, 64))
        layers.append(nn.ReLU())
        layers.append(nn.Linear(64, 32))
        layers.append(nn.ReLU())
        layers.append(nn.Linear(32, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.model(x))

X_train_tensor = torch.tensor(X_train)
Y_train_tensor = torch.tensor(Y_train).view(-1, 1)
X_val_tensor = torch.tensor(X_val)
Y_val_tensor = torch.tensor(Y_val).view(-1, 1)

model = MLP(X_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-2)

In [4]:
train_losses = []
val_losses = []
accuracies = []
f1_scores = []

best_val = float('inf')
best_epoch = 0
patience = 25
patience_counter = 0
best_model_weights = None

num_epochs = 5000
for epoch in range(num_epochs):
    model.train()

    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, Y_train_tensor)
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())


    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, Y_val_tensor)
        val_losses.append(val_loss.item())

        val_predictions = (val_outputs.view(-1) >= 0.5).float()
        accuracy = accuracy_score(Y_val, val_predictions.cpu().numpy())
        f1 = f1_score(Y_val, val_predictions.cpu().numpy())
        accuracies.append(accuracy)
        f1_scores.append(f1)

    if val_loss < best_val:
        best_val = val_loss
        best_model_weights = model.state_dict()
        patience_counter = 0
        best_epoch = epoch + 1
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch + 1}')
        break

    print(f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}, Accuracy: {accuracy:.2f}, F1 Score: {f1:.2f}')


model.load_state_dict(best_model_weights)
model.eval()
with torch.no_grad():
    val_outputs = model(X_val_tensor)
val_predictions = (val_outputs.view(-1) >= 0.5).float()
print(accuracy_score(Y_val, val_predictions.cpu().numpy()))
print(f1_score(Y_val, val_predictions.cpu().numpy()))

Epoch [1/5000], Train Loss: 0.6832, Val Loss: 0.6704, Accuracy: 0.76, F1 Score: 0.00
Epoch [2/5000], Train Loss: 0.6831, Val Loss: 0.6701, Accuracy: 0.76, F1 Score: 0.00
Epoch [3/5000], Train Loss: 0.6830, Val Loss: 0.6699, Accuracy: 0.76, F1 Score: 0.00
Epoch [4/5000], Train Loss: 0.6829, Val Loss: 0.6696, Accuracy: 0.76, F1 Score: 0.00
Epoch [5/5000], Train Loss: 0.6828, Val Loss: 0.6694, Accuracy: 0.76, F1 Score: 0.00
Epoch [6/5000], Train Loss: 0.6827, Val Loss: 0.6691, Accuracy: 0.76, F1 Score: 0.00
Epoch [7/5000], Train Loss: 0.6826, Val Loss: 0.6689, Accuracy: 0.76, F1 Score: 0.00
Epoch [8/5000], Train Loss: 0.6825, Val Loss: 0.6687, Accuracy: 0.76, F1 Score: 0.00
Epoch [9/5000], Train Loss: 0.6824, Val Loss: 0.6684, Accuracy: 0.76, F1 Score: 0.00
Epoch [10/5000], Train Loss: 0.6823, Val Loss: 0.6682, Accuracy: 0.76, F1 Score: 0.00
Epoch [11/5000], Train Loss: 0.6822, Val Loss: 0.6680, Accuracy: 0.76, F1 Score: 0.00
Epoch [12/5000], Train Loss: 0.6821, Val Loss: 0.6677, Accuracy

KeyboardInterrupt: 

In [ ]:
model.load_state_dict(best_model_weights)
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(train_losses, label='Train Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='orange')
plt.axvline(x=best_epoch, color='red', linestyle='--', label='Best Epoch')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Train and Validation Loss')
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(accuracies, label='Accuracy', color='green')
plt.axvline(x=best_epoch, color='red', linestyle='--', label='Best Epoch')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(f1_scores, label='F1 Score', color='red')
plt.axvline(x=best_epoch, color='red', linestyle='--', label='Best Epoch')
plt.xlabel('Epochs')
plt.ylabel('F1 Score')
plt.title('Validation F1 Score')
plt.legend()

plt.tight_layout()
plt.grid(True)
plt.show()
exit()

In [ ]:
import pickle
with open('model.pkl', 'wb') as file:
    pickle.dump(model, file)

In [7]:
import os
import shutil
if not os.path.exists('./data/ts_cut/HCPex/'):
    os.makedirs('./data/ts_cut/HCPex/')

np.save('./data/ts_cut/HCPex/predict.npy', np.concatenate([np.load(bnu_series_path.format(i)) for i in (1, 2)], axis=0))


In [ ]:
import numpy as np
import pandas as pd
import pickle

from scripts.data_utils import get_connectome
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_size, 128))
        layers.append(nn.ELU())
        layers.append(nn.Linear(128, 64))
        layers.append(nn.ELU())
        layers.append(nn.Linear(64, 32))
        layers.append(nn.ELU())
        layers.append(nn.Linear(32, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        # return self.model(x)
        return torch.sigmoid(self.model(x))

X = np.load('./data/ts_cut/HCPex/predict.npy')
X = get_connectome(X)
with open('model.pkl', 'rb') as file:
    model = pickle.load(file)

n_samples = X.shape[0]
X = X.reshape(n_samples, -1).astype('float32')
X_tensor = torch.tensor(X)

model.eval()
with torch.no_grad():
    y_pred = model(X_tensor)
    y_pred_binary = (y_pred.view(-1) >= 0.5).float()

y_pred = y_pred_binary.numpy()

solution = pd.DataFrame(data=y_pred, columns=['prediction'])
solution.to_csv('./solution.csv', index=False)

print(X.shape)
print(y_pred)
print(type(y_pred))

In [9]:
# build the .zip to submit
import zipfile
import datetime

# save source from previous cell into file
# will produce the correct result only in case of running previous cell just before
with open('run.py', 'w') as f_run:
    f_run.write(_ih[-2])

with open('run.sh', 'w') as f_run_sh:
    f_run_sh.write('export PATH=/usr/conda/bin:$PATH\npython run.py')

with open('train.py', 'w') as f_run:
    f_run.write('print("\\n".join(map(str, range(100))))')

with open('train.sh', 'w') as f_run_sh:
    f_run_sh.write('export PATH=/usr/conda/bin:$PATH\npython train.py')

with open('Makefile', 'w') as f_makefile:
    f_makefile.write('''all: build

build:
	@echo 'starting....'
	bash train.sh
run:
	bash run.sh
train:
	bash train.sh
''')

submission_zip = zipfile.ZipFile(f"submission-{datetime.datetime.now()}.zip".replace(':', '-').replace(' ', '-'), "w")
submission_zip.write('./Makefile', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('run.py', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('run.sh', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('train.py', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('train.sh', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('model.pkl', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('scripts', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('scripts/__init__.py', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('scripts/classification_models.py', compress_type=zipfile.ZIP_DEFLATED)
submission_zip.write('scripts/data_utils.py', compress_type=zipfile.ZIP_DEFLATED)

submission_zip.close()
